# Use Case 1: Loading and Cleaning the Chicago Crime Data

The goal here is to read the raw Chicago crime files, clean them up, add a few useful columns, and load everything into MySQL.

```
 CSV files ──► Pandas DataFrame ──► cleaning ──► new features ──► NumPy quality checks ──► MySQL
 (6 files)      (load + inspect)    (dates, NAs,   (Year, Month,     (% missing,             (normalized
                                     case, dupes)   DayOfWeek)        drop >50% cols)          schema)
```

This notebook produces two things that notebooks 02 to 04 rely on:
* `output/chicago_crime_clean.pkl`: the cleaned DataFrame (saved as a pickle so the datetime and bool types survive)
* the MySQL database `chicago_crime_db` with 6 tables

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

import crime_utils as cu

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

## 1. Load the data into a DataFrame

`iucr_code`, `case_number` and `fbi_code` look like numbers but they're really codes (`031A`, `08B` and so on), so I read them as strings. If pandas treated them as numbers it would quietly turn `0486` into `486`, and it would fail outright on `031A`.

In [2]:
crime_path = cu.data_file("crime")

# A plain pd.read_csv(crime_path) fails on this file with:
#   "ParserError: Expected 20 fields in line 498, saw 21"
# The last two lines each open a quote that never closes, e.g. "(41.76..., -87.64...)
# (looks like the export got cut off), so the parser glues them into one bad row.
# cu.read_csv_repaired closes those quotes before parsing.
raw, repaired_lines = cu.read_csv_repaired(
    crime_path, dtype={"iucr_code": str, "case_number": str, "fbi_code": str}, low_memory=False)
print(f"Lines repaired (unclosed quote): {repaired_lines}")

# clean up column names: strip spaces, lower case, snake_case
raw.columns = raw.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)

Lines repaired (unclosed quote): [498, 499]


### 1a. First 10 rows

In [3]:
display(raw.head(10))

,id,case_number,date,block,iucr_code,location_desc,arrest,domestic,beat_num,district_code,ward_no,community_code,fbi_code,x_coordinate,y_coordinate,year,date_of_update,latitude,longitude,location
0,10508693,HZ250496,5/3/2016,013XX S SAWYER AVE,486,APARTMENT,True,True,1022,10,24,29,08B,1154907.0,1893681.0,2016,5/10/2016,41.864073,-87.706819,"(41.864073157, -87.706818608)"
1,10508695,HZ250409,5/3/2016,061XX S DREXEL AVE,486,RESIDENCE,False,True,313,3,20,42,08B,1183066.0,1864330.0,2016,5/10/2016,41.782922,-87.604363,"(41.782921527, -87.60436317)"
2,10508697,HZ250503,5/3/2016,053XX W CHICAGO AVE,470,STREET,False,False,1524,15,37,25,24,1140789.0,1904819.0,2016,5/10/2016,41.894908,-87.758372,"(41.894908283, -87.758371958)"
3,10508698,HZ250424,5/3/2016,049XX W FULTON ST,460,SIDEWALK,False,False,1532,15,28,25,08B,1143223.0,1901475.0,2016,5/10/2016,41.885687,-87.749516,"(41.885686845, -87.749515983)"
4,10508699,HZ250455,5/3/2016,003XX N LOTUS AVE,820,RESIDENCE,False,True,1523,15,28,25,6,1139890.0,1901675.0,2016,5/10/2016,41.886297,-87.761751,"(41.886297242, -87.761750709)"
5,10508702,HZ250447,5/3/2016,082XX S MARYLAND AVE,041A,STREET,False,False,631,6,8,44,04B,1183336.0,1850642.0,2016,5/10/2016,41.745354,-87.603799,"(41.745354023, -87.603798903)"
6,10508703,HZ250489,5/3/2016,027XX S STATE ST,460,CHA HALLWAY/STAIRWELL/ELEVATOR,False,False,133,1,3,35,08B,1176730.0,1886544.0,2016,5/10/2016,41.844024,-87.626923,"(41.844023772, -87.626923253)"
7,10508704,HZ250514,5/3/2016,002XX E 46TH ST,460,RESIDENCE PORCH/HALLWAY,False,False,215,2,3,38,08B,1178514.0,1874573.0,2016,5/10/2016,41.811134,-87.620741,"(41.811133958, -87.62074077)"
8,10508709,HZ250523,5/3/2016,014XX W DEVON AVE,460,SIDEWALK,False,False,2432,24,40,1,08B,1165696.0,1942616.0,2016,5/10/2016,41.998131,-87.665814,"(41.99813061, -87.665814038)"
9,10508982,HZ250667,5/3/2016,069XX S ASHLAND AVE,486,STREET,False,True,735,7,17,67,08B,1166876.0,1858796.0,2016,5/10/2016,41.768097,-87.663879,"(41.768096835, -87.663878589)"


### 1b. Columns and data types
`df.info()` lists each column with its non-null count and dtype. A few things stand out:
* `date` and `date_of_update` came in as text, so they need converting to datetime.
* `arrest` and `domestic` were read as real booleans (pandas recognises `TRUE`/`FALSE`).
* `x_coordinate`, `y_coordinate`, `latitude` and `longitude` have some missing values.

In [4]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 498 entries, 0 to 497
Data columns (total 20 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              498 non-null    int64  
 1   case_number     498 non-null    str    
 2   date            498 non-null    str    
 3   block           498 non-null    str    
 4   iucr_code       498 non-null    str    
 5   location_desc   494 non-null    str    
 6   arrest          498 non-null    bool   
 7   domestic        498 non-null    bool   
 8   beat_num        498 non-null    int64  
 9   district_code   498 non-null    int64  
 10  ward_no         498 non-null    int64  
 11  community_code  498 non-null    int64  
 12  fbi_code        498 non-null    str    
 13  x_coordinate    469 non-null    float64
 14  y_coordinate    469 non-null    float64
 15  year            498 non-null    int64  
 16  date_of_update  498 non-null    str    
 17  latitude        469 non-null    float64
 18  l

### 1c. Number of rows and columns

In [5]:
n_rows, n_cols = raw.shape
print(f"Rows: {n_rows:,}   Columns: {n_cols}")

Rows: 498   Columns: 20


### 1d. Load the 5 lookup files
These are the tables the crime data points to through foreign keys.

The files I received don't line up perfectly with the data model, so I tidy the headers while loading:
* `iucr_codes.csv.csv` has the header `IUCR, PRIMARY_TYPE␣, DESCRIPTION, INDEX CODE` (there's a trailing space after PRIMARY_TYPE and a space inside INDEX CODE). I rename these to `iucr_code, primary_type, description, index_code`.
* In `chicago_ward_offices.csv` the key column is just `ward`, so I rename it to `ward_no`.

In [6]:
RENAME = {"iucr": "iucr_code", "ward": "ward_no"}

def load_lookup(key):
    d = pd.read_csv(cu.data_file(key), dtype=str)
    d.columns = (d.columns.str.strip().str.lower()
                   .str.replace(r"\s+", "_", regex=True))   # 'PRIMARY_TYPE ' -> 'primary_type'
    d = d.rename(columns=RENAME)
    return d.apply(lambda s: s.str.strip())          # strip whitespace in every cell

iucr_raw = load_lookup("iucr")
community_raw = load_lookup("community")
district_raw = load_lookup("district")
ward_raw = load_lookup("ward")
beat_raw = load_lookup("beat")

for name, d in [("iucr", iucr_raw), ("community", community_raw), ("district", district_raw),
                ("ward", ward_raw), ("beat", beat_raw)]:
    print(f"{name:<10} {d.shape[0]:>4} rows x {d.shape[1]:>2} cols   {list(d.columns)[:6]} ...")

iucr        410 rows x  4 cols   ['iucr_code', 'primary_type', 'description', 'index_code'] ...
community    77 rows x  7 cols   ['community_code', 'community_name', 'population', 'area_sqmile', 'area_sqkm', 'density_per_sqmi'] ...
district     22 rows x 15 cols   ['district_code', 'district_name', 'address', 'city', 'state', 'zip'] ...
ward         58 rows x 20 cols   ['ward_no', 'alderman', 'address', 'city', 'state', 'zipcode'] ...
beat        277 rows x  4 cols   ['district', 'sector', 'beat', 'beat_num'] ...


## 2. Clean the data

I do all the cleaning on a copy called `df` and leave `raw` alone, so I can compare before and after.

### 2a. Remove duplicates
* Exact duplicates, i.e. the same row appearing twice.
* Repeated `case_number`. This is the primary key so it has to be unique. If a case shows up more than once I keep the copy with the latest `date_of_update`.

In [7]:
df = raw.copy()

n_exact = df.duplicated().sum()
df = df.drop_duplicates()

# case_number is the primary key: tidy it, then keep the most recently updated copy of each case
df["case_number"] = df["case_number"].str.strip().str.upper()
df["_upd"], _ = cu.parse_mixed_dates(df["date_of_update"])
n_case_dup = df.duplicated("case_number").sum()
df = (df.sort_values("_upd").drop_duplicates("case_number", keep="last")
        .drop(columns="_upd").sort_index())

n_no_case = df["case_number"].isna().sum()
df = df[df["case_number"].notna()]
print(f"Exact duplicates removed: {n_exact} | duplicate case_number removed: {n_case_dup} | "
      f"rows with no case_number removed: {n_no_case}")

Exact duplicates removed: 0 | duplicate case_number removed: 0 | rows with no case_number removed: 0


### 2b. Convert `date` to datetime

The expected format is `5/3/2016 23:40`, i.e. `%m/%d/%Y %H:%M`. Exports often contain a mix of formats though, and a single `pd.to_datetime` call with one format would just turn everything else into `NaT` without saying anything. So instead I try a short list of known formats one after another (`cu.parse_mixed_dates`) and note which one worked for each row. Whatever doesn't match any of them is a real problem.

In [8]:
df["date_parsed"], df["date_format"] = cu.parse_mixed_dates(df["date"])
fmt_counts = df["date_format"].value_counts(dropna=False)
print("Which date formats were found:")
display(fmt_counts.rename("rows").to_frame())

Which date formats were found:


,rows
date_format,
%m/%d/%Y,498


In [9]:
bad_dates = df[df["date_parsed"].isna()]
n_bad_dates = len(bad_dates)
print(f"Values that could not be parsed: {n_bad_dates}")
display(bad_dates["date"].value_counts(dropna=False).head(10).rename("rows").to_frame())
df = df[df["date_parsed"].notna()].copy()

Values that could not be parsed: 0


,rows
date,


In [10]:
df["date"] = df.pop("date_parsed")
df = df.drop(columns="date_format")
df["date_of_update"], _ = cu.parse_mixed_dates(df["date_of_update"])
print(f"Date range: {df['date'].min()}  to  {df['date'].max()}")

Date range: 2012-01-01 00:00:00  to  2016-05-04 00:00:00


### 2c. Tidy up the text columns (whitespace and case)

`"APARTMENT"`, `"Apartment"` and `"  apartment "` mean the same thing, but pandas would count them as three different values. I strip spaces, collapse double spaces and upper-case every text column.

The IUCR codes need one more fix. The official list uses 4 characters (`0486`), but a lot of files drop the leading zero (`486`). I pad both the crime data and the lookup table with `zfill(4)` so the join works.

In [11]:
def standardize_text(s):
    return (s.astype("string").str.strip()
             .str.replace(r"\s+", " ", regex=True).str.upper())

before = df["location_desc"].nunique()
for col in ["block", "location_desc", "fbi_code", "iucr_code", "case_number"]:
    df[col] = standardize_text(df[col])
print(f"location_desc distinct values: {before} before → {df['location_desc'].nunique()} after")

df["iucr_code"] = df["iucr_code"].str.zfill(4)
iucr = iucr_raw.copy()
iucr["iucr_code"] = standardize_text(iucr["iucr_code"]).str.zfill(4)
for col in ["primary_type", "description", "index_code"]:
    iucr[col] = standardize_text(iucr[col])
iucr = iucr.drop_duplicates("iucr_code")

location_desc distinct values: 47 before → 47 after


Next I join the IUCR lookup to get `primary_type` (the crime category) and `description`. The main file only has the code; the category name is in `iucr_codes.csv`.

In [12]:
df = df.merge(iucr[["iucr_code", "primary_type", "description"]], on="iucr_code", how="left")

unknown_iucr = df["primary_type"].isna()
print(f"IUCR codes not found in lookup: {unknown_iucr.sum()}")
df["primary_type"] = df["primary_type"].fillna("UNKNOWN")
df["description"] = df["description"].fillna("UNKNOWN")

IUCR codes not found in lookup: 0


### 2d. Missing values in the key fields

There isn't one rule that fits every column, so I decided case by case. The general idea: don't fill in anything I can't back up.

| Field | What I did | Reason |
|---|---|---|
| `case_number`, `date`, `iucr_code` | drop the row | the primary key, the date and the crime code are needed for every analysis |
| `location_desc` | fill with `"UNKNOWN"` | keeps the row and makes the gap visible in charts |
| `latitude`/`longitude`/`x`/`y` | leave as `NULL` | not needed for counting, and I don't want to make up locations |

In [13]:
key_fields = ["case_number", "date", "iucr_code", "primary_type", "arrest", "domestic",
              "location_desc", "beat_num", "district_code", "ward_no", "community_code",
              "latitude", "longitude"]
missing_before = df[key_fields].isna().sum().rename("missing_before")

df = df.dropna(subset=["case_number", "date", "iucr_code"])
df["location_desc"] = df["location_desc"].fillna("UNKNOWN")

missing_after = df[key_fields].isna().sum().rename("missing_after")
display(pd.concat([missing_before, missing_after], axis=1))

,missing_before,missing_after
case_number,0,0
date,0,0
iucr_code,0,0
primary_type,0,0
arrest,0,0
domestic,0,0
location_desc,4,0
beat_num,0,0
district_code,0,0
ward_no,0,0


## 3. Add new columns
The `.dt` accessor only works on datetime columns, which is why the date conversion in 2b had to happen first. I add `year`, `month` and `day_of_week` (snake_case, to match the MySQL schema).

In [14]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.day_name()

display(df[["date", "year", "month", "day_of_week"]].head())

,date,year,month,day_of_week
0,2016-05-03,2016,5,Tuesday
1,2016-05-03,2016,5,Tuesday
2,2016-05-03,2016,5,Tuesday
3,2016-05-03,2016,5,Tuesday
4,2016-05-03,2016,5,Tuesday


## 4. Percentage of missing values per column (NumPy)

`df.isna().to_numpy()` gives a 2-D array of True/False values, where True means missing. True counts as 1 and False as 0, so taking `np.mean(..., axis=0)` down each column gives the fraction of missing values in that column. I run it on both the raw data and the cleaned data to compare.

In [15]:
def missing_pct_numpy(frame):
    mask = frame.isna().to_numpy()                    # shape (rows, cols), dtype=bool
    pct = np.mean(mask, axis=0) * 100                 # column-wise mean -> fraction missing
    return pd.Series(np.round(pct, 3), index=frame.columns)

missing_report = pd.DataFrame({
    "raw_missing_%": missing_pct_numpy(raw),
    "clean_missing_%": missing_pct_numpy(df.reindex(columns=raw.columns)),
}).sort_values("raw_missing_%", ascending=False)
display(missing_report)

,raw_missing_%,clean_missing_%
location,5.823,5.823
y_coordinate,5.823,5.823
longitude,5.823,5.823
latitude,5.823,5.823
x_coordinate,5.823,5.823
location_desc,0.803,0.000
case_number,0.000,0.000
id,0.000,0.000
block,0.000,0.000
date,0.000,0.000


### Dropping columns with more than 50% missing
`df.columns[pct > 50]` finds them and `df.drop(columns=...)` removes them.

In [16]:
THRESHOLD = 50
pct_now = missing_pct_numpy(df)
to_drop = pct_now[pct_now > THRESHOLD].index.tolist()
df = df.drop(columns=to_drop)
print(f"Columns with > {THRESHOLD}% missing: {to_drop or 'none'} | "
      f"highest missing % in any column: {pct_now.max():.2f}% ({pct_now.idxmax()})")

Columns with > 50% missing: none | highest missing % in any column: 5.82% (x_coordinate)


## 5. Load the cleaned data into MySQL

### 5a. Schema
The schema is in `sql/01_create_schema.sql` and follows the data model:

* `chicago_crime` is the fact table, with one row per incident.
* `iucr`, `city_community`, `district_ps_info`, `ward_office` and `police_beat_info` are the lookup tables, connected with foreign keys.
* `primary_type` is only stored in `iucr`, and we join when we need it.

### 5b. Writing with SQLAlchemy and PyMySQL
The lookup tables go in first and the fact table last, otherwise MySQL rejects crime rows that point to a lookup row that isn't there yet. The load cell empties the tables before writing (crimes first, then lookups), so it can be run again on its own.

In [17]:
from sqlalchemy import text

server = cu.get_engine(with_db=False)
with server.begin() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {cu.DB['database']} "
                      "CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci"))

engine = cu.get_engine()
n_stmt = cu.run_sql_file(engine, cu.SQL_DIR / "01_create_schema.sql")
print(f"Schema created ({n_stmt} statements)")

Schema created (14 statements)


Before loading, the lookup tables need a bit of prep: text like `"55,062"` has to become a number, and the duplicate wards and beats have to go because their key columns are primary keys.

In [18]:
def to_num(s, kind=float):
    v = pd.to_numeric(s.astype("string").str.replace(",", "", regex=False), errors="coerce")
    return v.round().astype("Int64") if kind is int else v

community = community_raw.copy()
community["community_code"] = to_num(community["community_code"], int)
community["population"] = to_num(community["population"], int)
for c in ["area_sqmile", "area_sqkm", "density_per_sqmi", "density_per_sqkm"]:
    community[c] = to_num(community[c])
community["community_name"] = standardize_text(community["community_name"])
community = community.dropna(subset=["community_code"]).drop_duplicates("community_code")

district = district_raw.copy()
district["district_code"] = to_num(district["district_code"], int)
for c in ["x_coordinate", "y_coordinate", "latitude", "longitude"]:
    district[c] = to_num(district[c])
district = district.dropna(subset=["district_code"]).drop_duplicates("district_code")

ward = ward_raw.copy()
ward["ward_no"] = to_num(ward["ward_no"], int)
# ward_no is the primary key: some wards are listed twice (old and new office), keep the last one
ward = ward.dropna(subset=["ward_no"]).drop_duplicates("ward_no", keep="last")

beat = beat_raw.copy()
for c in ["beat_num", "district", "sector", "beat"]:
    beat[c] = to_num(beat[c], int)
# beat_num is the primary key: beat 3100 appears three times (identical rows), keep one
beat = beat.dropna(subset=["beat_num"]).drop_duplicates("beat_num")

In [19]:
CRIME_COLS = ["id", "case_number", "date", "block", "iucr_code", "location_desc", "arrest",
              "domestic", "beat_num", "district_code", "ward_no", "community_code", "fbi_code",
              "x_coordinate", "y_coordinate", "year", "date_of_update", "latitude", "longitude",
              "location"]

crime_db = df[[c for c in CRIME_COLS if c in df.columns]].copy()
crime_db["arrest"] = crime_db["arrest"].astype(int)
crime_db["domestic"] = crime_db["domestic"].astype(int)
crime_db = crime_db.astype(object).where(crime_db.notna(), None)   # NaN -> SQL NULL

loads = [("iucr", iucr), ("city_community", community), ("district_ps_info", district),
         ("ward_office", ward), ("police_beat_info", beat), ("chicago_crime", crime_db)]

# Empty the tables first so this cell can be run again without duplicate-key errors.
# Crimes go first, otherwise the foreign keys stop the lookups from being emptied.
with engine.begin() as conn:
    for table, _ in reversed(loads):
        conn.execute(text(f"DELETE FROM {table}"))

for table, frame in loads:
    cols = pd.read_sql(text(f"SELECT * FROM {table} LIMIT 0"), engine).columns
    frame = frame[[c for c in frame.columns if c in cols]]
    frame.to_sql(table, engine, if_exists="append", index=False, chunksize=5000, method="multi")
    print(f"Loaded {len(frame):>7,} rows into {table}")

Loaded     410 rows into iucr
Loaded      77 rows into city_community
Loaded      22 rows into district_ps_info
Loaded      50 rows into ward_office
Loaded     275 rows into police_beat_info


Loaded     498 rows into chicago_crime


Quick check that the row counts in MySQL match the DataFrames.

In [20]:
counts = pd.read_sql(text("""
    SELECT 'chicago_crime' AS table_name, COUNT(*) AS rows_in_mysql FROM chicago_crime
    UNION ALL SELECT 'iucr', COUNT(*) FROM iucr
    UNION ALL SELECT 'city_community', COUNT(*) FROM city_community
    UNION ALL SELECT 'district_ps_info', COUNT(*) FROM district_ps_info
    UNION ALL SELECT 'ward_office', COUNT(*) FROM ward_office
    UNION ALL SELECT 'police_beat_info', COUNT(*) FROM police_beat_info"""), engine)
display(counts)
assert counts.loc[counts.table_name == "chicago_crime", "rows_in_mysql"].item() == len(df)
print("Row counts match")

,table_name,rows_in_mysql
0,chicago_crime,498
1,iucr,410
2,city_community,77
3,district_ps_info,22
4,ward_office,50
5,police_beat_info,275


Row counts match


## Questions

### Q1. How many unique crime types are in the dataset?
I'm taking "crime type" to mean `primary_type` (the IUCR category). The more detailed IUCR codes are shown as well.

In [21]:
known = df.loc[df["primary_type"] != "UNKNOWN", "primary_type"]
print(f"Unique crime types (primary_type): {known.nunique()}")
print(f"Unique IUCR codes (sub-types):     {df['iucr_code'].nunique()}")
print(f"Rows with an unrecognised code:    {(df['primary_type'] == 'UNKNOWN').sum()}")
print("\nCrime types:", ", ".join(sorted(known.unique())))

Unique crime types (primary_type): 23
Unique IUCR codes (sub-types):     97
Rows with an unrecognised code:    0

Crime types: ASSAULT, BATTERY, BURGLARY, CRIMINAL DAMAGE, CRIMINAL SEXUAL ASSAULT, CRIMINAL TRESPASS, DECEPTIVE PRACTICE, GAMBLING, HOMICIDE, INTERFERENCE WITH PUBLIC OFFICER, KIDNAPPING, MOTOR VEHICLE THEFT, NARCOTICS, OBSCENITY, OFFENSE INVOLVING CHILDREN, OTHER OFFENSE, PROSTITUTION, PUBLIC PEACE VIOLATION, ROBBERY, SEX OFFENSE, STALKING, THEFT, WEAPONS VIOLATION


### Q2. Are there any problems with the date formats, or obvious outliers?
Each check below is detection only: it counts the rows, it doesn't change them.

In [22]:
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 41.64, 42.03, -87.95, -87.52
has_coords = df["latitude"].notna() & df["longitude"].notna()
in_box = df["latitude"].between(LAT_MIN, LAT_MAX) & df["longitude"].between(LON_MIN, LON_MAX)
src_year = (raw.assign(case_number=raw["case_number"].str.strip().str.upper())
               .drop_duplicates("case_number", keep="last").set_index("case_number")["year"])
formats = ", ".join(f"{f} ({n})" for f, n in fmt_counts.items())
has_time = (df["date"].dt.time != pd.Timestamp(0).time()).sum()

checks = pd.DataFrame([
    ("date formats found", formats + f"; {has_time} rows have a time of day"),
    ("unparseable dates", n_bad_dates),
    ("dates in the future", (df["date"] > pd.Timestamp.today()).sum()),
    ("year column disagrees with date", (df["case_number"].map(src_year) != df["date"].dt.year).sum()),
    ("positive longitude (missing minus sign)", (df["longitude"] > 0).sum()),
    ("coordinates outside the Chicago box", (has_coords & ~in_box).sum()),
    ("truncated lines repaired when reading", len(repaired_lines)),
], columns=["check", "rows"]).set_index("check")
display(checks)

display(df[["year", "month", "beat_num", "district_code", "ward_no", "community_code",
            "latitude", "longitude"]].describe().T[["min", "max", "mean"]])

,rows
check,
date formats found,%m/%d/%Y (498); 0 rows have a time of day
unparseable dates,0
dates in the future,0
year column disagrees with date,0
positive longitude (missing minus sign),0
coordinates outside the Chicago box,0
truncated lines repaired when reading,2


,min,max,mean
year,2012.000000,2016.000000,2015.965863
month,1.000000,11.000000,4.821285
beat_num,111.000000,2535.000000,1115.927711
district_code,1.000000,25.000000,10.925703
ward_no,1.000000,50.000000,22.510040
community_code,1.000000,77.000000,37.112450
latitude,41.653250,42.019399,41.842657
longitude,-87.839659,-87.535758,-87.671066


Even if every value is formatted correctly, the data can still be lopsided. Here I look at how the crimes are spread across dates.

In [23]:
by_day = df["date"].dt.date.value_counts()
top2_share = by_day.head(2).sum() / len(df) * 100
print(f"Date range: {df['date'].min():%d %b %Y} to {df['date'].max():%d %b %Y}")
print(f"Distinct dates: {by_day.size} | busiest 2 dates hold {top2_share:.1f}% of all crimes:")
display(by_day.head(5).rename("crimes").to_frame())
display(df["year"].value_counts().sort_index().rename("crimes").to_frame().T)
if top2_share > 50:
    print("Note: the extract is concentrated on a few days in early May 2016; the older dates are crimes "
          "that were reported late, so it isn't a full history.")

Date range: 01 Jan 2012 to 04 May 2016
Distinct dates: 46 | busiest 2 dates hold 89.0% of all crimes:


,crimes
date,
2016-05-04,298
2016-05-03,145
2016-04-01,4
2015-01-01,2
2016-01-01,2


year,2012,2013,2014,2015,2016
crimes,2,1,1,4,490


Note: the extract is concentrated on a few days in early May 2016; the older dates are crimes that were reported late, so it isn't a full history.


## Save the outputs for the other notebooks

In [24]:
df.to_pickle(cu.CLEAN_CRIME_FILE)
community.to_pickle(cu.OUTPUT_DIR / "community_lookup.pkl")
print(f"Saved {len(df):,} clean rows to {cu.CLEAN_CRIME_FILE.name}")
print(f"Raw rows: {len(raw):,} | clean rows: {len(df):,} "
      f"({100 * (1 - len(df) / len(raw)):.2f}% removed)")

Saved 498 clean rows to chicago_crime_clean.pkl
Raw rows: 498 | clean rows: 498 (0.00% removed)
